## Imports

In [18]:
import argparse
import copy
import json
import random
import sys
from pathlib import Path
from typing import Optional
from types import SimpleNamespace

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

from preprocessing import preprocess_mindrove_data
from models import CNNLSTMClassifier
from task_sampler import FOMAMLTaskSampler
from fomaml_train import train_fomaml
from eval import evaluate, evaluate_full, eval_cm

## Load in Config file & Model

In [19]:
print("GLOBAL VARIABLES")
cfg = json.load(open("fomaml_config.json"), object_hook=lambda d: SimpleNamespace(**d))
cfg.seed = random.randint(1,1000000)
for var, val in vars(cfg).items():
    print(f"{var} : {val}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nGESTURE CLASSIFIER MODEL")
model_cfg = json.load(open("artifacts/metadata_v10.json"), object_hook=lambda d: SimpleNamespace(**d))
print(model_cfg)
model = CNNLSTMClassifier(
        input_size       = model_cfg.num_model_input_features,
        num_classes      = model_cfg.num_classes,
        conv_channels    = model_cfg.model_config.conv_channels,
        kernel_size      = model_cfg.model_config.kernel_size,
        lstm_hidden_size = model_cfg.model_config.lstm_hidden_size,
        lstm_num_layers  = model_cfg.model_config.lstm_num_layers,
        dropout          = model_cfg.model_config.dropout,
        bidirectional    = model_cfg.model_config.bidirectional
    )
state_dict = torch.load("artifacts/best_model_v10.pt",weights_only=True)
print(state_dict)
model.load_state_dict(state_dict)
# print(model)
# print(device)

GLOBAL VARIABLES
data_dir : subject_data
model_path : artifacts/best_model_v10.pt
model_metadata_path : artifacts/metadata_v10.json
meta_epochs : 100
inner_lr : 0.01
inner_steps : 3
outer_lr : 0.001
tasks_per_epoch : 50
k_shot : 5
q_query : 10
adapt_step : 10
adapt_lr : 0.001
seed : 978331

GESTURE CLASSIFIER MODEL
namespace(window_size=200, step_size=25, sample_rate_hz=500, min_segment_seconds=0.75, num_raw_channels=8, num_model_input_features=16, num_classes=8, raw_channel_columns=['Channel1', 'Channel2', 'Channel3', 'Channel4', 'Channel5', 'Channel6', 'Channel7', 'Channel8'], engineered_feature_names=['Channel1', 'Channel2', 'Channel3', 'Channel4', 'Channel5', 'Channel6', 'Channel7', 'Channel8', 'Channel1_delta', 'Channel2_delta', 'Channel3_delta', 'Channel4_delta', 'Channel5_delta', 'Channel6_delta', 'Channel7_delta', 'Channel8_delta'], feature_engineering=namespace(remove_window_dc_offset=True, add_delta_features=True), label_map=namespace(Extend=0, Fist=1, Flex=2, Pro=3, Radial=4

<All keys matched successfully>

### Freeze Layers for Fine Tuned Model

In [20]:
fine_tuned_model = copy.deepcopy(model)
# freeze cnn layer
for param in fine_tuned_model.cnn.parameters():
    param.requires_grad = False

# Load in Subject Data

### Load in all data
- Training data
- Fine Tuning Subject

In [21]:
data_dir = r"./mindrove_data"
ft_data_dir = r"./fine_tune_data"

data = preprocess_mindrove_data(data_dir)
data_sub = preprocess_mindrove_data(ft_data_dir)

X = data["X"]
y = data["y"]
del data

X_ft, X_ev, y_ft, y_ev = train_test_split(
    data_sub["X"], data_sub["y"], test_size=0.20, stratify=data_sub["y"],random_state=random.randint(1,1000000))
del data_sub


label_map = vars(model_cfg.raw_spec_label_map)
y = np.array([label_map[label] for label in y], dtype=np.int64)
y_ft = np.array([label_map[label] for label in y_ft], dtype=np.int64)
y_ev = np.array([label_map[label] for label in y_ev], dtype=np.int64)

print(X_ft.shape)
print(X_ft.shape)
print(X_ev.shape)


Number of CSV files found before exclusion: 60
Number of CSV files excluded: 0
Number of CSV files used: 60

Example file used: mindrove_data\2026-05-19T18-34-39.720895.csv

Sample rate: 500 Hz
Window size: 200 samples = 0.4 seconds
Step size: 25 samples = 0.05 seconds

raw_df shape: (1476620, 14)
Detected EMG channels: ['Channel1', 'Channel2', 'Channel3', 'Channel4', 'Channel5', 'Channel6', 'Channel7', 'Channel8']
Detected number of channels: 8

Label distribution including NaN pauses:
label
Extend     96176
Fist       92486
Flex       95566
Pro        89672
Radial     91156
Rest       96964
Sup       100258
Ulnar      97098
NaN       717244
Name: count, dtype: int64
Using new Mindrove row-level labels from the last CSV column.
NaN labels are pause/unmarked rows and will be ignored during window creation.
Using known Mindrove sampling rate: 500 Hz
Window size: 200 samples = 0.4 seconds
Step size: 25 samples = 0.05 seconds

Rows:
Total rows: 1476620
Labeled rows: 759376
Pause/unmarked 

### Quick Sanity Check

In [22]:
print("Y DIST")
values, counts = np.unique(y, return_counts=True)
print(dict(zip(values, counts)))

print("Y_FT DIST")
values, counts = np.unique(y_ft, return_counts=True)
print(dict(zip(values, counts)))


Y DIST
{0: 3357, 1: 3237, 2: 3346, 3: 3139, 4: 3184, 5: 3389, 6: 3513, 7: 3402}
Y_FT DIST
{0: 433, 1: 416, 2: 411, 3: 422, 4: 504, 5: 441, 6: 424, 7: 416}


### Turn Data in Tasks
- Training data
- Fine Tuning Subject

In [23]:
sampler = FOMAMLTaskSampler(
    X_train = X,  
    y_train = y,   
    X_tune  = X_ft,
    y_tune  = y_ft,
    cfg     = cfg,
    n_way   = None,
)

print(sampler)
print(sampler.tasks_per_epoch)  

[TaskSampler] classes      : [0, 1, 2, 3, 4, 5, 6, 7]
[TaskSampler] meta-train   | classes: 8 | total windows: 26567
[TaskSampler] meta-tune    | classes: 8 | total windows: 3467
[TaskSampler] episode      | 8-way 5-shot + 10-query per class
{'meta_train': 1672, 'meta_tune': 216}


### Prepare Data for Final Evaluation/ Training Accuracy

In [24]:
class EMGWindowsDataset(Dataset):
    """
    A PyTorch Dataset for EMG windows.
    """

    def __init__(
        self,
        X: np.ndarray,
        y: np.ndarray,
        augment: bool = False,
        noise_std: float = 0.01,
        gain_jitter_std: float = 0.05
    ):
        if len(X) != len(y):
            raise ValueError("X and y must have the same number of samples")

        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

        self.augment = augment
        self.noise_std = noise_std
        self.gain_jitter_std = gain_jitter_std

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].clone()
        y = self.y[idx]

        if self.augment:
            # Add very small Gaussian noise
            x = x + torch.randn_like(x) * self.noise_std

            # Add very small per-channel amplitude jitter
            gains = 1.0 + torch.randn(x.shape[1]) * self.gain_jitter_std
            gains = gains.to(x.device)
            x = x * gains.unsqueeze(0)

        return x, y
    

test_dataset = EMGWindowsDataset(
    X_ev,
    y_ev,
    augment=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

### Train Time!

In [25]:
fine_tuned_model = train_fomaml(
    model=fine_tuned_model,
    sampler=sampler,
    cfg=cfg,
    device=device
)

c:\Users\Ahuma\miniconda3\envs\ml\lib\site-packages\torch\nn\modules\module.py:810: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at C:\cb\pytorch_1000000000000\work\build\aten\src\ATen/core/TensorBody.h:494.)
  param_grad = param.grad


[Epoch   1/100] avg query loss: 3.3520 | avg eval loss: 1.9252
[Epoch   2/100] avg query loss: 2.6273 | avg eval loss: 1.5877
[Epoch   3/100] avg query loss: 1.9902 | avg eval loss: 1.2001
[Epoch   4/100] avg query loss: 1.4714 | avg eval loss: 1.0482
[Epoch   5/100] avg query loss: 1.2190 | avg eval loss: 0.9190
[Epoch   6/100] avg query loss: 0.9659 | avg eval loss: 0.8518
[Epoch   7/100] avg query loss: 0.7690 | avg eval loss: 0.8781
[Epoch   8/100] avg query loss: 0.6510 | avg eval loss: 0.9129
[Epoch   9/100] avg query loss: 0.6044 | avg eval loss: 0.7311
[Epoch  10/100] avg query loss: 0.5028 | avg eval loss: 0.7013
[Epoch  11/100] avg query loss: 0.4804 | avg eval loss: 0.5822
[Epoch  12/100] avg query loss: 0.4204 | avg eval loss: 0.6224
[Epoch  13/100] avg query loss: 0.4271 | avg eval loss: 0.4449
[Epoch  14/100] avg query loss: 0.3698 | avg eval loss: 0.5624
[Epoch  15/100] avg query loss: 0.3270 | avg eval loss: 0.4900
[Epoch  16/100] avg query loss: 0.3147 | avg eval loss:

In [26]:
from sklearn.metrics import classification_report
import torch.nn as nn

criterion = nn.CrossEntropyLoss(
    label_smoothing=0.0
)

print("ON !!BASE!! MODEL")
test_metrics = evaluate(model, test_loader, criterion, device)
print(
    f"Test Loss={test_metrics['loss']:.4f}, "
    f"Test Acc={test_metrics['accuracy']:.4f}, "
    f"Test MacroF1={test_metrics['macro_f1']:.4f}"
)

print("ON !!FINE TUNED!! MODEL")
test_metrics_ft = evaluate(fine_tuned_model, test_loader, criterion, device)
print(
    f"Test Loss={test_metrics_ft['loss']:.4f}, "
    f"Test Acc={test_metrics_ft['accuracy']:.4f}, "
    f"Test MacroF1={test_metrics_ft['macro_f1']:.4f}"
)

target_names = vars(model_cfg.label_map)
target_names = list(target_names.keys())

y_true, y_pred = evaluate_full(model, test_loader, device)
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(target_names))),
    target_names=target_names,
    digits=4,
    zero_division=0
))

__, y_pred = evaluate_full(fine_tuned_model, test_loader, device)
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(target_names))),
    target_names=target_names,    
    digits=4,
    zero_division=0
))


ON !!BASE!! MODEL


Test Loss=3.9973, Test Acc=0.5571, Test MacroF1=0.5601
ON !!FINE TUNED!! MODEL
Test Loss=0.1993, Test Acc=0.9308, Test MacroF1=0.9293
              precision    recall  f1-score   support

      Extend     0.9592    0.4352    0.5987       108
        Fist     0.3198    0.6058    0.4186       104
        Flex     0.4099    0.6408    0.5000       103
         Pro     0.9902    0.9619    0.9758       105
      Radial     0.9859    0.5556    0.7107       126
        Rest     0.9615    0.2252    0.3650       111
         Sup     0.2000    0.2170    0.2081       106
       Ulnar     0.6027    0.8462    0.7040       104

    accuracy                         0.5571       867
   macro avg     0.6787    0.5609    0.5601       867
weighted avg     0.6896    0.5571    0.5623       867

              precision    recall  f1-score   support

      Extend     0.8898    0.9722    0.9292       108
        Fist     1.0000    0.8846    0.9388       104
        Flex     0.8583    1.0000    0.9238       10

## Save Fine Tuned Model

In [27]:
torch.save(fine_tuned_model, r"artifacts/fine_tuned_model.pt")

# play example since subjects aren't id formally lol
meta = {
    "subject_id": -1,
    "subject_data_folder": "fine_tine_data"
}
# too lazy to write more :P

with open(r"artifacts/fine_tuned.json", "w") as f:
    json.dump(meta, f, indent=2)